# Read from Bronze Table

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col
from pyspark.sql.types import StringType

In [0]:
raw_df = spark.table("workspace.bronze.crm_sales_details")
display(raw_df.limit(50))

# Data Transformations

In [0]:
rename_mapping = {
    "sls_ord_num": "order_number",
    "sls_prd_key": "product_number",
    "sls_cust_id": "customer_id",
    "sls_order_dt": "order_date",
    "sls_ship_dt": "ship_date",
    "sls_due_dt": "due_date",
    "sls_sales": "sales_amount",
    "sls_quantity": "quantity",
    "sls_price": "price"
}

df = raw_df

# Trimming
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, F.trim(col(field.name)))

# Cast date columns to date type (from int)
df = (
    df
    .withColumn(
        "sls_order_dt",
        F.when(
            (col("sls_order_dt") == 0) | (F.length(col("sls_order_dt")) != 8),
            None
        ).otherwise(F.to_date(col("sls_order_dt").cast("string"), "yyyyMMdd"))
    )
    .withColumn(
        "sls_ship_dt",
        F.when(
            (col("sls_ship_dt") == 0) | (F.length(col("sls_ship_dt")) != 8),
            None
        ).otherwise(F.to_date(col("sls_ship_dt").cast("string"), "yyyyMMdd"))
    )
    .withColumn(
        "sls_due_dt",
        F.when(
            (col("sls_due_dt") == 0) | (F.length(col("sls_due_dt")) != 8),
            None
        ).otherwise(F.to_date(col("sls_due_dt").cast("string"), "yyyyMMdd"))
    )
)

# Data corrections for sls_sales, sls_price
df = (
    df
    .withColumn(
        "sls_sales",
        F.when(
            col("sls_sales").isNull() | (col("sls_sales") < 0) | (col("sls_sales") != col("sls_quantity") * F.abs("sls_price" )),
            col("sls_quantity") * F.abs("sls_price")
        )
        .otherwise(F.col("sls_sales"))
    )
)
df = (
    df
    .withColumn(
        "sls_price",
        F.when(
            col("sls_price").isNull() | (col("sls_price") <= 0),
            col("sls_sales") / F.nullif(F.abs("sls_quantity"), F.lit(0))
        )
        .otherwise(col("sls_price"))
    )
)

# Column names are not friendly
for old_name, new_name in rename_mapping.items():
    df = df.withColumnRenamed(old_name, new_name)

display(df.limit(50))

# Write into Silver Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_sales")

# Sanity checks of silver table

In [0]:
%sql
SELECT * FROM workspace.silver.crm_sales LIMIT 20;